In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
# Multiple Output Channel Cross-Correlation

def corr2d_multi_in(
    X: torch.Tensor, # [in, H, W]
    K: torch.Tensor, # [in, H, W]
) -> torch.Tensor:
    
    # 각 Channel 끼리 Convolution 연산 후 Sum
    channel_outputs: list[torch.Tensor] = [
        d2l.corr2d(
            X_channel,
            K_channel,
        )
        for X_channel, K_channel
        in zip(X, K) # 같은 Channel 끼리 매칭
    ]

    return torch.stack(
        channel_outputs,
        dim=0,
    ).sum(dim=0)
    
    
    
def corr2d_multi_in_out(
    X: torch.Tensor, # [in, H, W]
    K: torch.Tensor, # [out, in, H, W]
) -> torch.Tensor:

    output_channels: list[torch.Tensor] = [
        corr2d_multi_in(
            X,
            K_for_output,
        ) 
        for K_for_output in K
    ]

    return torch.stack(
        output_channels,
        dim=0,
    )

In [3]:
# 1x1 Convolution (Kernel Window size: [out, in, 1, 1])

def corr2d_multi_in_out_1x1(
    X: torch.Tensor, # [in, H, W] 
    K: torch.Tensor, # [out, in, H, W]
) -> torch.Tensor:
    
    input_channels, height, width = X.shape
    output_channels = K.shape[0]

    K_matrix = K.reshape(
        output_channels,
        input_channels,
    )
    
    X_matrix = X.reshape(
        input_channels,
        height * width,
    )
    
    # Y = K @ X 
    # Y.shape: [output_channels, height * width]
    Y = K_matrix @ X_matrix
    
    return Y.reshape(
        output_channels,
        height,
        width,
    )

In [4]:
X = torch.normal(
    mean=0, 
    std=1, 
    size=(3, 3, 3), # [in, H, W]
) 

K = torch.normal(
    mean=0,
    std=1,
    size=(2, 3, 1, 1), # [out, in, H, W]
)

Y_1x1 = corr2d_multi_in_out_1x1(
    X,
    K,
)
Y_general = corr2d_multi_in_out(
    X,
    K,
)

torch.testing.assert_close(
    Y_1x1,
    Y_general,
)

max_difference = (
    Y_1x1 - Y_general
).abs().max().item()

print("X shape:", X.shape)
print("K shape:", K.shape)
print("Y shape:", Y_1x1.shape)
print(
    "Maximum difference:",
    max_difference,
)

X shape: torch.Size([3, 3, 3])
K shape: torch.Size([2, 3, 1, 1])
Y shape: torch.Size([2, 3, 3])
Maximum difference: 0.0
